# Lobbying → Bill Alignment (end-to-end demo)

The capstone's core pipeline on real data:

1. Pull real **LDA lobbying filings**
2. **Extract bill references** from the issue descriptions (HR 824, S1001, ...)
3. **Fetch those bills** from Congress.gov
4. **Score semantic alignment** between the lobbying text and the bill text

> Needs `CONGRESS_API_KEY` in `.env` for steps 3–4. Steps 1–2 work without it.

In [1]:
# === Setup (run once) ===
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import requests
from analysis.bills import extract_bill_refs
from analysis.align import similarity
from extract.config import settings

print("Ready. Congress key present:", bool(settings.congress_api_key))

Ready. Congress key present: True


## 1–2. Pull lobbying filings + extract bill references (no key)

In [2]:
# === Steps 1-2: lobbying filings -> bill references ===
resp = requests.get(
    "https://lda.senate.gov/api/v1/filings/",
    params={"filing_year": 2024, "page_size": 25},
    timeout=30,
)
filings = resp.json()["results"]

# Find lobbying activities whose description names at least one bill.
hits = []
for f in filings:
    for act in f.get("lobbying_activities", []):
        desc = act.get("description") or ""
        refs = extract_bill_refs(desc)
        if refs:
            hits.append({
                "client": (f.get("client") or {}).get("name"),
                "issue": act.get("general_issue_code"),
                "desc": desc,
                "bills": [r.key for r in refs],
            })

print(f"{len(hits)} lobbying activities reference a bill.\n")
for h in hits[:5]:
    print(f"{h['client']}  [{h['issue']}]  -> {h['bills']}")
    print(f"   {h['desc'][:90]}\n")

2 lobbying activities reference a bill.

CHESAPEAKE CONSERVANCY  [NAT]  -> ['hr5045', 's2620']
   Support enactment of HR 5045, S.2620

CTIA - THE WIRELESS ASSOCIATION  [TEC]  -> ['hr3949']
   H.R.3949, End Cells in Cells Act, a bill to increase criminal penalties for contraband cel



## 3–4. Fetch the bill from Congress + score alignment (needs key)

Takes the first lobbying activity that references a bill, looks that bill up on
Congress.gov, and computes the alignment score between the lobbying text and the
bill's title/summary.

In [4]:
# === Steps 3-4: fetch bill text + alignment score (TF-IDF vs embeddings) ===
import re

KEY = settings.congress_api_key
assert KEY, "Set CONGRESS_API_KEY in .env to run this cell."

def fetch_bill(congress, bill_type, number):
    """Get a bill's title + latest summary text from Congress.gov."""
    base = f"https://api.congress.gov/v3/bill/{congress}/{bill_type}/{number}"
    p = {"api_key": KEY, "format": "json"}
    bill = requests.get(base, params=p, timeout=30).json().get("bill", {})
    title = bill.get("title", "")
    # Summaries live on a sub-endpoint.
    s = requests.get(base + "/summaries", params=p, timeout=30).json()
    summaries = s.get("summaries", [])
    summary = re.sub("<[^>]+>", " ", summaries[-1]["text"]) if summaries else ""
    return title, summary

# Score every lobbying activity that references a bill, with both backends.
print(f"{'CLIENT':<32} {'BILL':<8} {'TFIDF':>7} {'EMBED':>7}")
print("-" * 60)
for h in hits:
    m = re.match(r"([a-z]+)(\d+)", h["bills"][0])
    btype, bnum = m.group(1), int(m.group(2))
    title, summary = fetch_bill(118, btype, bnum)
    bill_text = f"{title}. {summary}"

    tfidf = similarity(h["desc"], bill_text, method="tfidf")
    embed = similarity(h["desc"], bill_text, method="embeddings")
    print(f"{h['client'][:31]:<32} {h['bills'][0]:<8} {tfidf:>7.3f} {embed:>7.3f}")


CLIENT                           BILL       TFIDF   EMBED
------------------------------------------------------------


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CHESAPEAKE CONSERVANCY           hr5045     0.000   0.094
CTIA - THE WIRELESS ASSOCIATION  hr3949     0.449   0.476
